In [ ]:
# =============================================================================
# 1. Import Libraries
# =============================================================================
print("Importing libraries...")
import pandas as pd
import h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import welch
import matplotlib.ticker as ticker
import os
from matplotlib.colors import LogNorm
from tqdm import tqdm


In [ ]:
# =============================================================================
# 2. Directory Setup
# =============================================================================
print("Setting up directories...")
# Create figs directory if it doesn't exist
figs_dir = "figs"
if not os.path.exists(figs_dir):
    os.makedirs(figs_dir)


In [ ]:
# =============================================================================
# 3. File Paths and Parameters
# =============================================================================
print("Setting up file paths and parameters...")
# File paths
file_name_eq = r"/users/230442014/archive/STEAD_dataset/chunk2.hdf5"
csv_file_eq = r"/users/230442014/archive/STEAD_dataset/chunk2.csv"

file_name_noise = r"/users/230442014/archive/STEAD_dataset/chunk1.hdf5"
csv_file_noise = r"/users/230442014/archive/STEAD_dataset/chunk1.csv"


In [ ]:
# =============================================================================
# 4. Processing Mode Configuration
# =============================================================================
print("Configuring processing mode...")
# Processing mode
MODE = 'test'  # 'test' or 'prod'
if MODE == 'test':
    chunksize = 1000  # Smaller chunk size for testing
    nrows = 1000     # Limit number of rows for testing
    
else:  # prod mode
    chunksize = 100000  # Larger chunk size for production
    nrows = None       # No row limit
    # Production mode filters (minimal or none)  


In [ ]:
# =============================================================================
# 5. Data Loading and Chunking
# =============================================================================
print("Loading and chunking data...")
# Initialize data readers
chunks_eq = pd.read_csv(csv_file_eq, chunksize=chunksize, nrows=nrows)
chunks_noise = pd.read_csv(csv_file_noise, chunksize=chunksize, nrows=nrows)

# Calculate total number of chunks
if MODE == 'test':
    total_chunks = min(nrows // chunksize + (1 if nrows % chunksize else 0), 
                      len(list(pd.read_csv(csv_file_eq, chunksize=chunksize, nrows=nrows))))
else:
    # For production, estimate total chunks from file size
    import os
    file_size = os.path.getsize(csv_file_eq)
    estimated_rows = file_size / 1000  # Rough estimate: 1KB per row
    total_chunks = (estimated_rows + chunksize - 1) // chunksize


In [ ]:
# =============================================================================
# 6. Data Filtering Configuration
# =============================================================================
print("Setting up data filtering...")
EQ_FILTERS = {
        'trace_category': 'earthquake_local',
        'source_distance_km': 30,  # <= 20 km
        'source_magnitude': (1, 3)  # between 1 and 3
    }

# EQ_FILTERS = {
#         'trace_category': 'earthquake_local',
#         'source_distance_km': None,  # No distance limit
#         'source_magnitude': None     # No magnitude limit
#     }


In [ ]:
# =============================================================================
# 7. Plotting Configuration
# =============================================================================
print("Configuring plotting settings...")
# Plotting parameters
plotting = True     # Toggle all plotting on/off
quiet = False    # If True, skip waveform+PSD plots
plot_aggregated_only = True  # If True, only plot aggregated PSD, skip individual waveform plots


In [ ]:
# =============================================================================
# 8. Data Storage Initialization
# =============================================================================
print("Initializing data storage...")
# Initialize storage variables
max_amp_e_eq = []
max_amp_n_eq = []
max_amp_z_eq = []
max_amp_e_noise = []
max_amp_n_noise = []
max_amp_z_noise = []
psd_data_eq = []  # [(f, Pxx_e, Pxx_n, Pxx_z), ...]
psd_data_noise = []  # [(f, Pxx_e, Pxx_n, Pxx_z), ...]

# Store all PSDs for aggregation
all_noise_psd_e = []
all_noise_psd_n = []
all_noise_psd_z = []
all_noise_freqs = []
all_noise_traces = []  # Store trace names for noise
all_eq_psd_e = []
all_eq_psd_n = []
all_eq_psd_z = []
all_eq_freqs = []
all_eq_traces = []  # Store trace names and info for earthquakes
all_eq_info = []  # Store magnitude and distance info

# Dictionary to store noise PSDs for reuse
noise_psd_cache = {}  # {trace_name: (f, Pxx_e, Pxx_n, Pxx_z)}


In [ ]:
# =============================================================================
# 9. Data Processing Functions
# =============================================================================
print("Setting up data processing functions...")
def create_aggregated_psd(
    all_noise_psd_e, all_noise_psd_n, all_noise_psd_z,
    all_eq_psd_e, all_eq_psd_n, all_eq_psd_z,
    all_noise_freqs, all_eq_freqs,
    all_noise_traces, all_eq_traces,
    all_eq_info,
    figs_dir
):
    """Create and save aggregated PSD plots in both log-log and linear scales
    
    Parameters:
    -----------
    all_noise_psd_e, all_noise_psd_n, all_noise_psd_z : list
        Lists of PSD arrays for noise data (E, N, Z components)
    all_eq_psd_e, all_eq_psd_n, all_eq_psd_z : list
        Lists of PSD arrays for earthquake data (E, N, Z components)
    all_noise_freqs, all_eq_freqs : list
        Lists of frequency arrays for noise and earthquake data
    all_noise_traces, all_eq_traces : list
        Lists of trace names for noise and earthquake data
    all_eq_info : list
        List of dictionaries containing magnitude and distance info for earthquakes
    figs_dir : str
        Directory path to save the figures
    """
    print("Creating aggregated PSD plots...")
    if len(all_noise_psd_e) > 0 and len(all_eq_psd_e) > 0:
        # Convert lists to arrays
        all_noise_psd_e_array = np.array(all_noise_psd_e)
        all_noise_psd_n_array = np.array(all_noise_psd_n)
        all_noise_psd_z_array = np.array(all_noise_psd_z)
        all_eq_psd_e_array = np.array(all_eq_psd_e)
        all_eq_psd_n_array = np.array(all_eq_psd_n)
        all_eq_psd_z_array = np.array(all_eq_psd_z)
        
        # Use the frequency arrays (they should all be the same within each dataset)
        freqs_noise = all_noise_freqs[0]
        freqs_eq = all_eq_freqs[0]
        
        # Calculate mean PSDs
        mean_noise_psd_e = np.mean(all_noise_psd_e_array, axis=0)
        mean_noise_psd_n = np.mean(all_noise_psd_n_array, axis=0)
        mean_noise_psd_z = np.mean(all_noise_psd_z_array, axis=0)
        mean_eq_psd_e = np.mean(all_eq_psd_e_array, axis=0)
        mean_eq_psd_n = np.mean(all_eq_psd_n_array, axis=0)
        mean_eq_psd_z = np.mean(all_eq_psd_z_array, axis=0)
        
        # Create two figures - one for log-log and one for linear scales
        fig_log, (ax1_log, ax2_log) = plt.subplots(1, 2, figsize=(20, 8), dpi=150)
        fig_lin, (ax1_lin, ax2_lin) = plt.subplots(1, 2, figsize=(20, 8), dpi=150)
        
        # Function to plot PSDs with given axes
        def plot_psds(ax1, ax2, is_log=False):
            # Plot noise PSDs
            if is_log:
                ax1.loglog(freqs_noise, mean_noise_psd_e, label='E component', color='C0')
                ax1.loglog(freqs_noise, mean_noise_psd_n, label='N component', color='C1')
                ax1.loglog(freqs_noise, mean_noise_psd_z, label='Z component', color='C2')
                ax2.loglog(freqs_eq, mean_eq_psd_e, label='E component', color='C0')
                ax2.loglog(freqs_eq, mean_eq_psd_n, label='N component', color='C1')
                ax2.loglog(freqs_eq, mean_eq_psd_z, label='Z component', color='C2')
            else:
                ax1.plot(freqs_noise, mean_noise_psd_e, label='E component', color='C0')
                ax1.plot(freqs_noise, mean_noise_psd_n, label='N component', color='C1')
                ax1.plot(freqs_noise, mean_noise_psd_z, label='Z component', color='C2')
                ax2.plot(freqs_eq, mean_eq_psd_e, label='E component', color='C0')
                ax2.plot(freqs_eq, mean_eq_psd_n, label='N component', color='C1')
                ax2.plot(freqs_eq, mean_eq_psd_z, label='Z component', color='C2')
            
            # Add shaded regions for standard deviation
            std_noise_psd_e = np.std(all_noise_psd_e_array, axis=0)
            std_noise_psd_n = np.std(all_noise_psd_n_array, axis=0)
            std_noise_psd_z = np.std(all_noise_psd_z_array, axis=0)
            std_eq_psd_e = np.std(all_eq_psd_e_array, axis=0)
            std_eq_psd_n = np.std(all_eq_psd_n_array, axis=0)
            std_eq_psd_z = np.std(all_eq_psd_z_array, axis=0)
            
            # Plot shaded regions
            for ax, mean_psds, std_psds, freqs, title, traces, info in zip(
                [ax1, ax2],
                [[mean_noise_psd_e, mean_noise_psd_n, mean_noise_psd_z],
                 [mean_eq_psd_e, mean_eq_psd_n, mean_eq_psd_z]],
                [[std_noise_psd_e, std_noise_psd_n, std_noise_psd_z],
                 [std_eq_psd_e, std_eq_psd_n, std_eq_psd_z]],
                [freqs_noise, freqs_eq],
                ['Noise', 'Earthquake'],
                [all_noise_traces, all_eq_traces],
                [None, all_eq_info]
            ):
                # Plot mean and std for each component
                for i, (mean, std) in enumerate(zip(mean_psds, std_psds)):
                    ax.fill_between(freqs, mean - std, mean + std, 
                                  color=f'C{i}', alpha=0.2)
                
                # Set plot properties
                ax.set_xlabel('Frequency (Hz)')
                ax.set_ylabel('PSD (counts²/Hz)')
                
                # Create detailed title
                if title == 'Noise':
                    title_text = f'Aggregated PSD - {title}\nTraces: {", ".join(traces[:3])}...'
                else:
                    mag_dist_info = "\n".join([f"M{info['magnitude']:.1f} @ {info['distance']:.1f}km" 
                                             for info in info[:3]])
                    title_text = f'Aggregated PSD - {title}\nTraces: {", ".join(traces[:3])}...\n{mag_dist_info}'
                
                ax.set_title(title_text)
                ax.grid(True, which='both', ls='--', lw=0.5)
                ax.legend()
        
        # Create both plots
        plot_psds(ax1_log, ax2_log, is_log=True)
        plot_psds(ax1_lin, ax2_lin, is_log=False)
        
        # Save both figures
        plt.figure(fig_log.number)
        plt.tight_layout()
        plt.savefig(os.path.join(figs_dir, 'aggregated_psd_comparison_loglog.png'), dpi=300, bbox_inches='tight')
        plt.close(fig_log)
        
        plt.figure(fig_lin.number)
        plt.tight_layout()
        plt.savefig(os.path.join(figs_dir, 'aggregated_psd_comparison_linear.png'), dpi=300, bbox_inches='tight')
        plt.close(fig_lin)
        
        print(f"Created aggregated PSD comparison plots with {len(all_noise_psd_e)} noise records and {len(all_eq_psd_e)} earthquake records")

def filter_data(chunk_eq, chunk_noise, mode='test', eq_filters=None):
    """Filter earthquake and noise data based on specified criteria
    
    Parameters:
    -----------
    chunk_eq : pandas.DataFrame
        Chunk of earthquake data
    chunk_noise : pandas.DataFrame
        Chunk of noise data
    mode : str
        Processing mode ('test' or 'prod')
    eq_filters : dict
        Dictionary containing earthquake filtering criteria
        
    Returns:
    --------
    tuple
        (filtered_eq_chunk, filtered_noise_chunk, all_noise_data)
    """
    if mode == 'test':
        filtered_eq = chunk_eq[
            (chunk_eq.trace_category == eq_filters['trace_category']) &
            (chunk_eq.source_distance_km <= eq_filters['source_distance_km']) &
            (chunk_eq.source_magnitude > eq_filters['source_magnitude'][0]) &
            (chunk_eq.source_magnitude < eq_filters['source_magnitude'][1])
        ]
    else:
        filtered_eq = chunk_eq[chunk_eq.trace_category == eq_filters['trace_category']]
    
    # Get all noise data for aggregated PSD
    all_noise_data = chunk_noise[chunk_noise.trace_category == 'noise']
    
    # Sample noise data to match earthquake data size for individual comparisons
    filtered_noise = all_noise_data.sample(n=len(filtered_eq), random_state=42) if not filtered_eq.empty else all_noise_data
    
    return filtered_eq, filtered_noise, all_noise_data

def compute_psd(data, fs):
    """Compute PSD using classical periodogram method for all components of seismic data
    
    Parameters:
    -----------
    data : numpy.ndarray
        Seismic data array with shape (n_samples, 3) for E, N, Z components
    fs : float
        Sampling frequency
        
    Returns:
    --------
    tuple
        (frequencies, PSD_E, PSD_N, PSD_Z)
    """
    n = data.shape[0]
    
    # Compute FFT for each component
    fft_e = np.fft.rfft(data[:, 0])
    fft_n = np.fft.rfft(data[:, 1])
    fft_z = np.fft.rfft(data[:, 2])
    
    # Compute frequencies
    f = np.fft.rfftfreq(n, 1/fs)
    
    # Compute PSD (periodogram)
    Pxx_e = np.abs(fft_e)**2 / (fs * n)
    Pxx_n = np.abs(fft_n)**2 / (fs * n)
    Pxx_z = np.abs(fft_z)**2 / (fs * n)
    
    return f, Pxx_e, Pxx_n, Pxx_z

def plot_comparison(t_eq, t_noise, data_eq, data_noise, 
                   f_eq, f_noise, Pxx_eq, Pxx_noise,
                   trace_name_eq, trace_name_noise, eq_info,
                   figs_dir, quiet=False):
    """Plot comparison between earthquake and noise data
    
    Parameters:
    -----------
    t_eq, t_noise : numpy.ndarray
        Time arrays for earthquake and noise data
    data_eq, data_noise : numpy.ndarray
        Seismic data arrays for earthquake and noise
    f_eq, f_noise : numpy.ndarray
        Frequency arrays for PSD computation
    Pxx_eq, Pxx_noise : tuple
        PSD arrays for earthquake and noise (E, N, Z components)
    trace_name_eq, trace_name_noise : str
        Names of the traces
    eq_info : pandas.Series
        Earthquake information
    figs_dir : str
        Directory to save figures
    quiet : bool
        If True, skip plotting
    """
    if quiet:
        return
        
    fig, axes = plt.subplots(5, 2, figsize=(16, 15), dpi=150)
    
    # Get earthquake info for title
    eq_title = f'Earthquake\n{trace_name_eq}\nM{eq_info.source_magnitude:.1f} @ {eq_info.source_distance_km:.1f}km'
    noise_title = f'Noise\n{trace_name_noise}'
    
    # Earthquake waveforms
    labels = ['E', 'N', 'Z']
    for i, ax in enumerate(axes[:3, 0]):
        ax.plot(t_eq, data_eq[:, i], 'k', lw=0.5)
        ax.set_ylabel(f'{labels[i]} counts')
        ax.grid(True)
        if i == 0:
            ax.set_title(eq_title)

    # Noise waveforms
    for i, ax in enumerate(axes[:3, 1]):
        ax.plot(t_noise, data_noise[:, i], 'k', lw=0.5)
        ax.set_ylabel(f'{labels[i]} counts')
        ax.grid(True)
        if i == 0:
            ax.set_title(noise_title)

    # Combined traces
    ax4_eq = axes[3, 0]
    ax4_eq.plot(t_eq, data_eq[:, 0], label='E', color='C0', lw=0.5)
    ax4_eq.plot(t_eq, data_eq[:, 1], label='N', color='C1', lw=0.5)
    ax4_eq.plot(t_eq, data_eq[:, 2], label='Z', color='C2', lw=0.5)
    ax4_eq.set_ylabel('Counts')
    ax4_eq.legend()
    ax4_eq.grid(True)
    ax4_eq.set_title(eq_title)

    ax4_noise = axes[3, 1]
    ax4_noise.plot(t_noise, data_noise[:, 0], label='E', color='C0', lw=0.5)
    ax4_noise.plot(t_noise, data_noise[:, 1], label='N', color='C1', lw=0.5)
    ax4_noise.plot(t_noise, data_noise[:, 2], label='Z', color='C2', lw=0.5)
    ax4_noise.set_ylabel('Counts')
    ax4_noise.legend()
    ax4_noise.grid(True)
    ax4_noise.set_title(noise_title)

    # PSDs
    ax5_eq = axes[4, 0]
    ax5_eq.plot(f_eq, Pxx_eq[0], label='E')
    ax5_eq.plot(f_eq, Pxx_eq[1], label='N')
    ax5_eq.plot(f_eq, Pxx_eq[2], label='Z')
    ax5_eq.set_xlabel('Frequency (Hz)')
    ax5_eq.set_ylabel('PSD (counts²/Hz)')
    ax5_eq.legend()
    ax5_eq.grid(True, which='both', ls='--', lw=0.5)
    ax5_eq.set_title(eq_title)

    ax5_noise = axes[4, 1]
    ax5_noise.plot(f_noise, Pxx_noise[0], label='E')
    ax5_noise.plot(f_noise, Pxx_noise[1], label='N')
    ax5_noise.plot(f_noise, Pxx_noise[2], label='Z')
    ax5_noise.set_xlabel('Frequency (Hz)')
    ax5_noise.set_ylabel('PSD (counts²/Hz)')
    ax5_noise.legend()
    ax5_noise.grid(True, which='both', ls='--', lw=0.5)
    ax5_noise.set_title(noise_title)

    plt.tight_layout()
    
    # Save the figure
    fig_path = os.path.join(figs_dir, f'comparison_{trace_name_eq}.png')
    plt.savefig(fig_path, dpi=300, bbox_inches='tight')
    plt.close(fig)


In [ ]:
# =============================================================================
# 10. Main Processing Loop
# =============================================================================
print("Starting main processing loop...")
# Create progress bar for chunks
with tqdm(total=total_chunks, desc=f"Processing chunks ({MODE} mode)") as pbar:
    for chunk_eq, chunk_noise in zip(chunks_eq, chunks_noise):
        # Filter data
        filtered_eq, filtered_noise, all_noise_data = filter_data(
            chunk_eq, chunk_noise, mode=MODE, eq_filters=EQ_FILTERS
        )
        
        if filtered_eq.empty:
            print("No events found in this chunk")
            pbar.update(1)
            continue

        ev_list_eq = filtered_eq['trace_name'].tolist()
        ev_list_noise = filtered_noise['trace_name'].tolist()

        # Process both earthquake and noise data
        with h5py.File(file_name_eq, 'r') as dtfl_eq, h5py.File(file_name_noise, 'r') as dtfl_noise:
            # Process earthquake data
            print("\nProcessing earthquake data:")
            for trace_name_eq in tqdm(ev_list_eq, desc="Earthquake traces", leave=False):
                ds_eq = dtfl_eq.get(f"data/{trace_name_eq}")
                
                if ds_eq is None:
                    continue

                # Process earthquake data
                data_eq = np.array(ds_eq)
                fs_eq = float(ds_eq.attrs.get('sampling_rate', 100.0))
                
                # Compute PSDs for earthquake data
                f_eq, Pxx_e_eq, Pxx_n_eq, Pxx_z_eq = compute_psd(data_eq, fs_eq)
                
                # Print shapes and some statistics
                print(f"\nEarthquake trace: {trace_name_eq}")
                print(f"Signal length: {len(data_eq)}")
                print(f"Frequency vector shape: {f_eq.shape}")
                print(f"PSD shapes - E: {Pxx_e_eq.shape}, N: {Pxx_n_eq.shape}, Z: {Pxx_z_eq.shape}")
                print(f"Frequency range: {f_eq[0]:.2f} Hz to {f_eq[-1]:.2f} Hz")
                print(f"Frequency resolution: {f_eq[1]-f_eq[0]:.2f} Hz")

            # Process noise data
            print("\nProcessing noise data:")
            for trace_name_noise in tqdm(ev_list_noise, desc="Noise traces", leave=False):
                ds_noise = dtfl_noise.get(f"data/{trace_name_noise}")
                
                if ds_noise is None:
                    continue

                # Process noise data
                data_noise = np.array(ds_noise)
                fs_noise = float(ds_noise.attrs.get('sampling_rate', 100.0))
                
                # Compute PSDs for noise data
                f_noise, Pxx_e_noise, Pxx_n_noise, Pxx_z_noise = compute_psd(data_noise, fs_noise)
                
                # Print shapes and some statistics
                print(f"\nNoise trace: {trace_name_noise}")
                print(f"Signal length: {len(data_noise)}")
                print(f"Frequency vector shape: {f_noise.shape}")
                print(f"PSD shapes - E: {Pxx_e_noise.shape}, N: {Pxx_n_noise.shape}, Z: {Pxx_z_noise.shape}")
                print(f"Frequency range: {f_noise[0]:.2f} Hz to {f_noise[-1]:.2f} Hz")
                print(f"Frequency resolution: {f_noise[1]-f_noise[0]:.2f} Hz")

        # Update chunk progress bar
        pbar.update(1)


In [ ]:
# =============================================================================
# 11. Final Processing and Results
# =============================================================================
print("Performing final processing and generating results...")
# Convert amplitude lists to arrays for further stats
max_amp_e_eq = np.array(max_amp_e_eq)
max_amp_n_eq = np.array(max_amp_n_eq)
max_amp_z_eq = np.array(max_amp_z_eq)
max_amp_e_noise = np.array(max_amp_e_noise)
max_amp_n_noise = np.array(max_amp_n_noise)
max_amp_z_noise = np.array(max_amp_z_noise)

# Create final aggregated PSD plot with all collected data
create_aggregated_psd(
    all_noise_psd_e, all_noise_psd_n, all_noise_psd_z,
    all_eq_psd_e, all_eq_psd_n, all_eq_psd_z,
    all_noise_freqs, all_eq_freqs,
    all_noise_traces, all_eq_traces,
    all_eq_info,
    figs_dir
)


